# NARR–PRISM Phase-2 stochastic residual refinement

This parameterized notebook is the NARR–PRISM front end for all four refinement heads: diffusion UNet, diffusion Transformer, flow-matching UNet, and flow-matching Transformer. It follows the Phase-1/Phase-2 structure of `SA_downscaling_refinement_T2_ACCESS-CM2_static.ipynb`, while delegating working training and inference to shared repository code.

> **Scientific-use boundary:** `SMOKE_TEST=True` uses a tiny synthetic fixture only to check implementation mechanics. Its fields and metrics are not NARR–PRISM validation and must not be presented as scientific skill. Production mode never substitutes synthetic data or weights.

## 1. Environment and repository paths

Run this notebook with the existing `Prithvi` mamba environment. No package installation or environment mutation is performed here.

In [ ]:
import json
import os
import random
import subprocess
import sys
from pathlib import Path

import numpy as np
import torch
import yaml

REPO_ROOT = Path.cwd()
while REPO_ROOT != REPO_ROOT.parent and not (REPO_ROOT / 'pyproject.toml').exists():
    REPO_ROOT = REPO_ROOT.parent
if not (REPO_ROOT / 'pyproject.toml').exists():
    raise RuntimeError('Run the notebook from within the granite-wxc repository.')
NARR_PRISM_DIR = REPO_ROOT / 'examples' / 'NARR_PRISM'
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
if str(NARR_PRISM_DIR) not in sys.path:
    sys.path.insert(0, str(NARR_PRISM_DIR))
print(f'Repository: {REPO_ROOT}')
print(f'Python: {sys.executable}')
print(f'Torch: {torch.__version__}; CUDA available: {torch.cuda.is_available()}')

## 2. Configuration selection and reproducibility

Edit `REFINEMENT_CONFIG` to switch heads. `PREDICTION_SPLIT` selects the held-out `validation` dates or the later `inference` dates; validation is the default for scientific comparison. `REFINEMENT_CHECKPOINT` is optional for a new training run and required for production prediction. `RESUME_FROM_CHECKPOINT=True` resumes Phase 2 without reinitializing either phase.

In [ ]:
def _env_flag(name, default=False):
    return os.environ.get(name, str(int(default))).strip().lower() in {'1', 'true', 'yes', 'on'}

# ========================== USER PARAMETERS ==========================
REFINEMENT_CONFIG = os.environ.get(
    'NARR_PRISM_REFINEMENT_CONFIG',
    'examples/NARR_PRISM/NARR_PRISM_diffusion_unet.yaml',
)
PHASE1_CHECKPOINT = os.environ.get(
    'NARR_PRISM_PHASE1_CHECKPOINT',
    'examples/NARR_PRISM/experiments/checkpoints/narr_prism_California/last.ckpt',
)
REFINEMENT_CHECKPOINT = os.environ.get('NARR_PRISM_REFINEMENT_CHECKPOINT') or None
RESUME_CHECKPOINT = os.environ.get('NARR_PRISM_RESUME_CHECKPOINT') or REFINEMENT_CHECKPOINT
ENSEMBLE_SIZE = int(os.environ.get('NARR_PRISM_ENSEMBLE_SIZE', '2'))
SMOKE_TEST = _env_flag('NARR_PRISM_SMOKE_TEST', False)
PREDICTION_SPLIT = os.environ.get(
    'NARR_PRISM_PREDICTION_SPLIT', 'validation'
).strip().lower()
if PREDICTION_SPLIT not in {'validation', 'inference'}:
    raise ValueError(
        'NARR_PRISM_PREDICTION_SPLIT must be validation or inference, '
        f'got {PREDICTION_SPLIT!r}'
    )
OUTPUT_DIR = os.environ.get(
    'NARR_PRISM_REFINEMENT_OUTPUT_DIR',
    'examples/NARR_PRISM/experiments/refinement_notebook',
)
PHASE1_DAILY_DIR = os.environ.get(
    'NARR_PRISM_PHASE1_DAILY_DIR',
    (
        'examples/NARR_PRISM/experiments/inference_output/validation/'
        'narr_prism_California'
        if PREDICTION_SPLIT == 'validation'
        else 'examples/NARR_PRISM/experiments/inference_output/'
        'narr_prism_California'
    ),
)
SEED = int(os.environ.get('NARR_PRISM_REFINEMENT_SEED', '1234'))
DEVICE = os.environ.get(
    'NARR_PRISM_REFINEMENT_DEVICE', 'cuda:0' if torch.cuda.is_available() else 'cpu'
)
RUN_TRAINING = _env_flag('NARR_PRISM_RUN_TRAINING', False)
RUN_INFERENCE = _env_flag('NARR_PRISM_RUN_INFERENCE', False)
RUN_EVALUATION = _env_flag('NARR_PRISM_RUN_EVALUATION', False)
RESUME_FROM_CHECKPOINT = _env_flag('NARR_PRISM_RESUME', False)
# =====================================================================

def repo_path(value):
    path = Path(value).expanduser()
    return path if path.is_absolute() else REPO_ROOT / path

CONFIG_PATH = repo_path(REFINEMENT_CONFIG)
PHASE1_PATH = repo_path(PHASE1_CHECKPOINT)
REFINEMENT_PATH = repo_path(REFINEMENT_CHECKPOINT) if REFINEMENT_CHECKPOINT else None
RESUME_PATH = repo_path(RESUME_CHECKPOINT) if RESUME_CHECKPOINT else None
OUTPUT_PATH = repo_path(OUTPUT_DIR)
OUTPUT_PATH.mkdir(parents=True, exist_ok=True)
PHASE1_DAILY_PATH = repo_path(PHASE1_DAILY_DIR)

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
print(json.dumps({
    'config': str(CONFIG_PATH), 'phase1_checkpoint': str(PHASE1_PATH),
    'refinement_checkpoint': str(REFINEMENT_PATH) if REFINEMENT_PATH else None,
    'resume_checkpoint': str(RESUME_PATH) if RESUME_PATH else None,
    'ensemble_size': ENSEMBLE_SIZE, 'smoke_test': SMOKE_TEST,
    'prediction_split': PREDICTION_SPLIT,
    'output_dir': str(OUTPUT_PATH), 'seed': SEED, 'device': DEVICE,
    'phase1_daily_dir': str(PHASE1_DAILY_PATH), 'run_evaluation': RUN_EVALUATION,
}, indent=2))

## 3. Configuration and checkpoint compatibility checks

The selected YAML is parsed through the shared schema. Production mode requires the real Phase-1 checkpoint and never falls back to random or synthetic weights. The shared CLI performs the detailed tensor and pipeline-contract checks when it loads the model.

In [ ]:
from granitewxc.refinement.config import resolve_refinement_config
from granitewxc.utils.config import get_config

if not CONFIG_PATH.is_file():
    raise FileNotFoundError(f'Refinement YAML not found: {CONFIG_PATH}')
raw_config = yaml.safe_load(CONFIG_PATH.read_text())
config = get_config(str(CONFIG_PATH))
refinement = resolve_refinement_config(config)
output_variables = list(config.data.output_vars)
if output_variables != ['ppt', 'tmax', 'tmin']:
    raise ValueError(f'Unexpected NARR–PRISM output order: {output_variables}')
if not refinement.is_active:
    raise ValueError('The selected configuration does not enable Phase-2 refinement.')

describe_command = [
    sys.executable, str(NARR_PRISM_DIR / 'narr_prism_refinement.py'), 'describe',
    '--config', str(CONFIG_PATH), '--phase1-checkpoint', str(PHASE1_PATH),
]
subprocess.run(describe_command, cwd=REPO_ROOT, check=True)

if not SMOKE_TEST:
    try:
        phase1_exists = PHASE1_PATH.is_file()
    except OSError as exc:
        raise FileNotFoundError(f'Cannot access Phase-1 checkpoint {PHASE1_PATH}: {exc}') from exc
    if not phase1_exists:
        raise FileNotFoundError(
            f'Production mode requires the real Phase-1 checkpoint: {PHASE1_PATH}. '
            'Set SMOKE_TEST=True only for the explicitly synthetic CI check.'
        )
    if RUN_INFERENCE and (REFINEMENT_PATH is None or not REFINEMENT_PATH.is_file()):
        raise FileNotFoundError('RUN_INFERENCE requires an existing REFINEMENT_CHECKPOINT.')
print(f'Active head: {refinement.type}; output order: {output_variables}')

## 4. Dataset, dates, normalization, units, and grid inspection

These values are displayed before any training. Residual statistics must be fitted only from the configured training split. The held-out validation dates are used for model comparison; the later inference dates are a distinct prediction period. Neither prediction split is used to fit normalization or residual statistics.

In [ ]:
data_summary = {
    'case_name': raw_config.get('case_name'),
    'input_vars': raw_config['data'].get('input_vars', []),
    'output_vars': raw_config['data'].get('output_vars', []),
    'target_variables': raw_config['data'].get('target_variables', []),
    'dates': raw_config.get('dates', {}),
    'spatial_subset': raw_config['data'].get('spatial_subset', {}),
    'crop': [raw_config['data'].get('train_crop_size_lat'), raw_config['data'].get('train_crop_size_lon')],
    'stride': [raw_config['data'].get('training_tile_stride_lat'), raw_config['data'].get('training_tile_stride_lon')],
    'halo': [raw_config['data'].get('training_halo_lat'), raw_config['data'].get('training_halo_lon')],
    'normalization': raw_config.get('normalization', {}),
    'predictands': raw_config.get('predictands', {}),
    'physical_units': {'ppt': 'mm/day', 'tmax': 'degC', 'tmin': 'degC'},
}
print(json.dumps(data_summary, indent=2))

## 5. Load and freeze Phase 1; inspect a deterministic baseline batch and residual target

Production inspection calls the same shared data/model builders as the CLI. It checks the frozen parameter state, runs one deterministic batch, and constructs the exact normalized residual target.

In [ ]:
model = None
baseline_batch = None
baseline_output = None
baseline_residual = None
baseline_valid = None
if not SMOKE_TEST:
    from narr_prism_refinement import build_model
    from narr_prism_training import get_dataloaders

    train_loader, validation_loader = get_dataloaders(str(CONFIG_PATH), config)
    baseline_batch = next(iter(train_loader))
    baseline_batch = {
        key: value.to(DEVICE) if torch.is_tensor(value) else value
        for key, value in baseline_batch.items()
    }
    model, phase1_fingerprint = build_model(
        config, str(CONFIG_PATH), str(PHASE1_PATH), torch.device(DEVICE)
    )
    model.initialize_from_batch(baseline_batch).eval()
    if not model.phase1_frozen or any(p.requires_grad for p in model.phase1.parameters()):
        raise RuntimeError('Phase 1 is not completely frozen in the default Phase-2 mode.')
    with torch.no_grad():
        baseline_output, baseline_normalized, _ = model.run_phase1(baseline_batch)
        baseline_residual, baseline_valid = model.target_space.residual_target(
            baseline_batch['y'], baseline_normalized,
            scaler_offset=baseline_batch.get('__output_scaler_offset', baseline_batch.get('__scaler_offset')),
        )
    print(model.describe())
    print(f'x={tuple(baseline_batch["x"].shape)}, y={tuple(baseline_batch["y"].shape)}')
    print(f'residual valid fraction={float(baseline_valid.float().mean()):.6f}')
else:
    print('Production Phase-1 loading skipped: explicit synthetic smoke mode is active.')

In [ ]:
if baseline_residual is not None:
    import matplotlib.pyplot as plt

    valid_residual = baseline_residual[baseline_valid]
    print({
        'mean': float(valid_residual.mean()), 'std': float(valid_residual.std()),
        'min': float(valid_residual.min()), 'max': float(valid_residual.max()),
    })
    fig, axes = plt.subplots(2, 3, figsize=(13, 8), constrained_layout=True)
    for index, name in enumerate(output_variables):
        axes[0, index].imshow(baseline_output[0, index].detach().cpu(), origin='lower')
        axes[0, index].set_title(f'Phase 1: {name}')
        axes[1, index].imshow(baseline_residual[0, index].detach().cpu(), origin='lower', cmap='RdBu_r')
        axes[1, index].set_title(f'Target residual: {name}')
    plt.show()

## 6. Refinement-head initialization

The production model above initializes the selected head from the real conditioning width. The head is the only trainable component by default.

In [ ]:
if model is not None:
    frozen_count = sum(p.numel() for p in model.phase1.parameters() if not p.requires_grad)
    trainable_count = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print({'frozen_phase1_parameters': frozen_count, 'trainable_parameters': trainable_count})

## 7. Phase 2 training and Resume support

Set `RUN_TRAINING=True` for production training. The command calls `narr_prism_refinement.py train`; no training implementation is hidden in notebook cells.

In [ ]:
train_command = [
    sys.executable, str(NARR_PRISM_DIR / 'narr_prism_refinement.py'), 'train',
    '--config', str(CONFIG_PATH), '--phase1-checkpoint', str(PHASE1_PATH),
]
if RESUME_FROM_CHECKPOINT:
    train_command.append('--resume')
    if RESUME_PATH is not None:
        train_command.append(str(RESUME_PATH))
print('Training command:', ' '.join(train_command))
if RUN_TRAINING:
    if SMOKE_TEST:
        raise RuntimeError('RUN_TRAINING is a production action and cannot be combined with SMOKE_TEST.')
    subprocess.run(train_command, cwd=REPO_ROOT, check=True)
else:
    print('Production training not requested in this execution.')

## 8. Lightweight smoke training, backward pass, sampling, and diagnostics

In explicit smoke mode, the reusable helper parses the selected real YAML and exercises its head with a tiny local Phase-1 fixture. It writes loss/gradient diagnostics, deterministic and refined fields, process states, metrics, and an ensemble NetCDF.

In [ ]:
smoke_result = None
if SMOKE_TEST:
    from refinement_smoke import run_smoke

    smoke_result = run_smoke(
        CONFIG_PATH, output_dir=OUTPUT_PATH, ensemble_size=ENSEMBLE_SIZE,
        seed=SEED, device=DEVICE,
    )
    print(json.dumps(smoke_result, indent=2, sort_keys=True))
    SMOKE_PREDICTION_PATH = Path(smoke_result['artifacts']['netcdf'])
else:
    SMOKE_PREDICTION_PATH = None
    print('Synthetic smoke harness skipped in production mode.')

## 9. Split-aware stochastic ensemble prediction

Set `RUN_INFERENCE=True` and supply `REFINEMENT_CHECKPOINT` to generate an ensemble for `PREDICTION_SPLIT` through the shared CLI. Validation products are isolated under `daily_predictions/validation/<case>`; inference products retain `daily_predictions/<case>`. Each product stores the refined ensemble mean, individual members, spread, deterministic baseline, and authenticated static support mask. PRISM truth is read separately by the evaluator; normalized residuals remain diagnostic model-space quantities.

In [ ]:
DAILY_OUTPUT_ROOT = OUTPUT_PATH / 'daily_predictions'
DAILY_SPLIT_ROOT = (
    DAILY_OUTPUT_ROOT / 'validation'
    if PREDICTION_SPLIT == 'validation'
    else DAILY_OUTPUT_ROOT
)
DAILY_OUTPUT_DIR = DAILY_SPLIT_ROOT / raw_config['case_name']
infer_command = [
    sys.executable, str(NARR_PRISM_DIR / 'narr_prism_refinement.py'), 'infer',
    '--config', str(CONFIG_PATH), '--phase1-checkpoint', str(PHASE1_PATH),
    '--ensemble-size', str(ENSEMBLE_SIZE), '--seed', str(SEED),
    '--output', str(DAILY_OUTPUT_ROOT), '--split', PREDICTION_SPLIT,
]
if REFINEMENT_PATH is not None:
    infer_command.extend(['--refinement-checkpoint', str(REFINEMENT_PATH)])
print('Inference command:', ' '.join(infer_command))
if RUN_INFERENCE:
    if SMOKE_TEST:
        raise RuntimeError('RUN_INFERENCE is a production action and cannot be combined with SMOKE_TEST.')
    subprocess.run(infer_command, cwd=REPO_ROOT, check=True)
    daily_files = sorted(DAILY_OUTPUT_DIR.glob('*_refined_*.nc'))
    if not daily_files:
        raise RuntimeError(f'Inference produced no daily files under {DAILY_OUTPUT_DIR}')
    print(f'Production daily files: {len(daily_files)} under {DAILY_OUTPUT_DIR}')
else:
    print('Production inference not requested in this execution.')

## 10. Evaluation: deterministic-versus-refined metrics, distributions, and extremes

The same selected-split samples are used for Phase 1, Phase 2, and truth. Smoke metrics below operate on the tiny aggregate smoke artifact and are implementation diagnostics only. Production prediction writes one canonical NetCDF per day; `evaluate_refinement.py` streams those files and the matching deterministic daily products without constructing an aggregate file. Use `PREDICTION_SPLIT=validation` for held-out scientific comparison; the inference period remains distinct. Set `RUN_EVALUATION=True` and point `PHASE1_DAILY_DIR` at the matching split's deterministic daily directory.

In [ ]:
import xarray as xr

EVALUATION_OUTPUT_PATH = OUTPUT_PATH / 'evaluation'
evaluate_command = [
    sys.executable, str(NARR_PRISM_DIR / 'evaluate_refinement.py'),
    '--config', str(CONFIG_PATH),
    '--phase1-dir', str(PHASE1_DAILY_PATH),
    '--method', f'{refinement.type}={DAILY_OUTPUT_DIR}',
    '--output-dir', str(EVALUATION_OUTPUT_PATH),
    '--split', PREDICTION_SPLIT,
]
print('Streaming evaluation command:', ' '.join(evaluate_command))
scientific_validation_completed = False
scientific_validation_report = None
if RUN_EVALUATION:
    if SMOKE_TEST:
        raise RuntimeError('RUN_EVALUATION is a production action and cannot be combined with SMOKE_TEST.')
    if not PHASE1_DAILY_PATH.is_dir():
        raise FileNotFoundError(f'Deterministic daily directory not found: {PHASE1_DAILY_PATH}')
    if not DAILY_OUTPUT_DIR.is_dir():
        raise FileNotFoundError(f'Refined daily directory not found: {DAILY_OUTPUT_DIR}')
    subprocess.run(evaluate_command, cwd=REPO_ROOT, check=True)
    selected_dates = raw_config['dates'][PREDICTION_SPLIT]
    report_stem = (
        f"{raw_config['case_name']}_refinement_"
        f"{str(selected_dates['start']).replace('-', '')}_"
        f"{str(selected_dates['end']).replace('-', '')}_metrics.json"
    )
    scientific_validation_report = EVALUATION_OUTPUT_PATH / report_stem
    if not scientific_validation_report.is_file():
        raise RuntimeError(
            'Evaluation returned successfully without writing the expected report: '
            f'{scientific_validation_report}'
        )
    completed_report = json.loads(scientific_validation_report.read_text())
    expected_range = {
        'start': str(selected_dates['start']),
        'end': str(selected_dates['end']),
    }
    actual_range = completed_report.get('date_range', {})
    if any(actual_range.get(key) != value for key, value in expected_range.items()):
        raise RuntimeError(
            f'Evaluation report date range {actual_range} does not match '
            f'{PREDICTION_SPLIT} dates {expected_range}'
        )
    if not completed_report.get('metrics'):
        raise RuntimeError(f'Evaluation report contains no metrics: {scientific_validation_report}')
    scientific_validation_completed = True

evaluation = {}
if SMOKE_TEST and SMOKE_PREDICTION_PATH is not None and SMOKE_PREDICTION_PATH.is_file():
    with xr.open_dataset(SMOKE_PREDICTION_PATH) as dataset:
        loaded = dataset.load()
    for name in output_variables:
        truth = loaded[f'{name}_truth'].values
        deterministic = loaded[name].values
        refined_name = f'{name}_refined' if f'{name}_refined' in loaded else f'{name}_ensemble_mean'
        refined_field = loaded[refined_name].values
        valid = np.isfinite(truth) & np.isfinite(deterministic) & np.isfinite(refined_field)
        def summarize(field):
            error = field[valid] - truth[valid]
            correlation = np.corrcoef(field[valid], truth[valid])[0, 1] if valid.sum() > 1 else np.nan
            return {
                'bias': float(np.mean(error)), 'absolute_bias': float(abs(np.mean(error))),
                'mae': float(np.mean(np.abs(error))), 'rmse': float(np.sqrt(np.mean(error ** 2))),
                'correlation': float(correlation),
                'q01_error': float(np.quantile(field[valid], .01) - np.quantile(truth[valid], .01)),
                'q05_error': float(np.quantile(field[valid], .05) - np.quantile(truth[valid], .05)),
                'q95_error': float(np.quantile(field[valid], .95) - np.quantile(truth[valid], .95)),
                'q99_error': float(np.quantile(field[valid], .99) - np.quantile(truth[valid], .99)),
            }
        evaluation[name] = {'phase1': summarize(deterministic), 'refined': summarize(refined_field)}
        if name == 'ppt':
            wet_truth = truth[valid] >= 0.1
            for label, field in [('phase1', deterministic), ('refined', refined_field)]:
                wet = field[valid] >= 0.1
                evaluation[name][label].update({
                    'wet_day_frequency': float(wet.mean()),
                    'wet_day_precision': float((wet & wet_truth).sum() / max(1, wet.sum())),
                    'wet_day_recall': float((wet & wet_truth).sum() / max(1, wet_truth.sum())),
                    'q999': float(np.quantile(field[valid], .999)),
                    'maximum': float(np.max(field[valid])),
                })
    if {'tmax_refined', 'tmin_refined'} <= set(loaded.data_vars):
        evaluation['tasmin_gt_tasmax_rate'] = float(
            np.nanmean(loaded['tmin_refined'].values > loaded['tmax_refined'].values)
        )
    evaluation['mode'] = 'synthetic_smoke'
    evaluation['scientific_validation'] = False
    metrics_path = OUTPUT_PATH / 'notebook_metrics.json'
    metrics_path.write_text(json.dumps(evaluation, indent=2, sort_keys=True) + '\n')
    print(json.dumps(evaluation, indent=2))
elif SMOKE_TEST:
    print(f'No smoke prediction file available yet: {SMOKE_PREDICTION_PATH}')
else:
    print(f'Production evaluation input is the daily directory: {DAILY_OUTPUT_DIR}')

## 11. Spatial maps and distribution diagnostics

Maps use consistent scales for Phase 1, refined output, and truth, plus common difference scales. The smoke helper also saves process-state panels for clean/noised or flow-interpolated residuals, model-estimated clean residual, sampled residual, baseline, refined field, and target.

In [ ]:
if SMOKE_TEST and SMOKE_PREDICTION_PATH is not None and SMOKE_PREDICTION_PATH.is_file():
    import matplotlib.pyplot as plt

    for name in output_variables:
        truth = loaded[f'{name}_truth'].values
        phase1 = loaded[name].values
        refined_key = f'{name}_refined' if f'{name}_refined' in loaded else f'{name}_ensemble_mean'
        refined_field = loaded[refined_key].values
        truth_mean = np.nanmean(truth, axis=0)
        phase1_mean = np.nanmean(phase1, axis=0)
        refined_mean = np.nanmean(refined_field, axis=0)
        field_min = np.nanmin([truth_mean, phase1_mean, refined_mean])
        field_max = np.nanmax([truth_mean, phase1_mean, refined_mean])
        difference_limit = np.nanmax(np.abs([phase1_mean - truth_mean, refined_mean - truth_mean]))
        fig, axes = plt.subplots(2, 3, figsize=(14, 8), constrained_layout=True)
        for axis, field, title in zip(axes[0], [phase1_mean, refined_mean, truth_mean], ['Phase 1 mean', 'Refined mean', 'Truth mean']):
            image = axis.imshow(field, origin='lower', vmin=field_min, vmax=field_max)
            axis.set_title(title); fig.colorbar(image, ax=axis, shrink=.75)
        phase1_rmse = np.sqrt(np.nanmean((phase1 - truth) ** 2, axis=0))
        refined_rmse = np.sqrt(np.nanmean((refined_field - truth) ** 2, axis=0))
        for axis, field, title, cmap, limit in [
            (axes[1, 0], phase1_mean - truth_mean, 'Phase 1 − truth', 'RdBu_r', difference_limit),
            (axes[1, 1], refined_mean - truth_mean, 'Refined − truth', 'RdBu_r', difference_limit),
            (axes[1, 2], phase1_rmse - refined_rmse, 'RMSE improvement', 'RdBu_r', None),
        ]:
            kwargs = {'vmin': -limit, 'vmax': limit} if limit is not None else {}
            image = axis.imshow(field, origin='lower', cmap=cmap, **kwargs)
            axis.set_title(title); fig.colorbar(image, ax=axis, shrink=.75)
        figure_path = OUTPUT_PATH / f'{name}_phase1_vs_refined.png'
        fig.savefig(figure_path, dpi=130)
        plt.close(fig)
    print(f'Saved smoke maps under {OUTPUT_PATH}')
elif not SMOKE_TEST:
    print('Production maps and distribution diagnostics are emitted by the streaming evaluator.')

## 12. Saving artifacts and next steps

Phase-2 checkpoints are kept separate from `last.ckpt`. Predictions, metrics, process diagnostics, and maps are written below `OUTPUT_DIR`. The final scientific-validation flag is evidence based: it becomes true only after the streaming evaluator completes and its expected split-specific metrics report is verified. Prediction-only production runs and smoke output never support an improvement claim.

In [ ]:
artifacts = sorted(str(path) for path in OUTPUT_PATH.rglob('*') if path.is_file())
print(json.dumps({
    'mode': 'synthetic_smoke' if SMOKE_TEST else 'production',
    'refinement_type': refinement.type,
    'prediction_split': PREDICTION_SPLIT,
    'output_dir': str(OUTPUT_PATH),
    'artifacts': artifacts,
    'scientific_validation': scientific_validation_completed,
    'scientific_validation_report': (
        str(scientific_validation_report) if scientific_validation_report else None
    ),
}, indent=2))